In [2]:
import pandas as pd

df = pd.read_csv("drone_paths_raw.csv")

# Filter to paths with n_cells > 1, then top 200 by danger_removed
# df_filtered = (df[df["n_cells"] > 1]
#                  .sort_values("danger_removed", ascending=False)
#                  .head(200))
# Weighted combo — tune the weights to taste
df["score"] = df["danger_removed"] * 0.5/93 + df["coverage_km2"] * 0.5/13000
df_filtered = df[df["n_cells"] > 1].sort_values("score", ascending=False).head(200)

df_filtered.to_csv("drone_paths_filtered.csv", index=False)
print(df_filtered[["n_cells","path_km","danger_removed","coverage_km2"]].head(20))

         n_cells  path_km  danger_removed  coverage_km2
1449635        7   37.071        0.720495         175.0
2936719        7   37.071        0.720495         175.0
2936717        7   37.071        0.720495         175.0
336890         7   37.071        0.720495         175.0
893253         7   37.071        0.720495         175.0
2030138        7   37.071        0.720495         175.0
2096367        7   37.071        0.720495         175.0
1886947        7   37.071        0.720495         175.0
2255767        7   37.071        0.720495         175.0
2255750        7   37.071        0.720495         175.0
2255612        7   37.071        0.720495         175.0
2936627        7   37.071        0.720495         175.0
337018         7   37.071        0.720495         175.0
1886130        7   37.071        0.720495         175.0
2836450        7   37.071        0.720495         175.0
2426330        7   37.071        0.720495         175.0
2096360        7   37.071        0.720495       

In [3]:
df_raw = pd.read_csv("drone_paths_raw.csv")
print(df_raw.sort_values("danger_removed", ascending=False).head(5)[
    ["n_cells", "path_km", "danger_removed", "path_str"]
])

         n_cells  path_km  danger_removed  \
1449635        7   37.071        0.720495   
2936719        7   37.071        0.720495   
2936717        7   37.071        0.720495   
336890         7   37.071        0.720495   
893253         7   37.071        0.720495   

                                          path_str  
1449635  10,34;11,34;12,35;12,36;11,36;10,36;10,35  
2936719  15,21;14,21;13,21;13,20;13,19;14,19;15,20  
2936717  15,21;14,21;13,21;13,20;14,19;15,19;15,20  
336890          7,50;7,51;8,52;9,52;9,51;9,50;8,50  
893253   13,38;14,38;15,39;15,40;14,40;13,40;13,39  


In [4]:
import numpy as np, pandas as pd

df_raw = pd.read_csv("drone_paths_raw.csv")
best = df_raw.iloc[0]
path = [tuple(int(x) for x in p.split(",")) for p in best["path_str"].split(";")]

# Load danger
fire_df = pd.read_csv("fire_risk_5km.csv", index_col=0)
animal_df = pd.read_csv("animal_value_5km.csv", index_col=0)

print("Path cells and their danger values:")
for r, c in path:
    print(f"  cell ({r},{c}): fire={fire_df.iloc[r,c]:.3f}, animal={animal_df.iloc[r,c]:.3f}")

Path cells and their danger values:
  cell (14,56): fire=0.324, animal=78817.252
  cell (13,55): fire=0.283, animal=81238.626
  cell (12,54): fire=0.196, animal=79420.559
  cell (11,53): fire=0.230, animal=77954.087
  cell (11,54): fire=0.197, animal=81261.589
  cell (12,55): fire=0.285, animal=82576.561
  cell (13,56): fire=0.330, animal=83541.742


In [5]:
import pandas as pd

df = pd.read_csv("drone_paths_filtered.csv")

# re-sort by danger_removed instead of avg_path_danger
df_by_danger = df.sort_values("danger_removed", ascending=False).reset_index(drop=True)
df_by_danger.to_csv("drone_paths_by_danger.csv", index=False)

In [6]:
# ── Re-sort drone_paths_raw.csv by danger_removed ────────────────────────────
# Run this in a Jupyter notebook cell after the generator has finished.
# Produces a new filtered CSV ranked by total danger removed instead of
# avg_path_danger.
 
import pandas as pd
 
df_raw = pd.read_csv("drone_paths_raw.csv")
 
def canonical(path_str):
    cells = sorted(path_str.split(";"))
    return ";".join(cells)
 
df_raw["_canon"] = df_raw["path_str"].apply(canonical)
 
df_by_danger = (df_raw
                .sort_values("danger_removed", ascending=False)
                .drop_duplicates(subset="_canon")
                .drop(columns="_canon")
                .head(200)
                .reset_index(drop=True))
 
df_by_danger.to_csv("drone_paths_by_danger.csv", index=False)
 
print(f"Saved {len(df_by_danger)} paths to drone_paths_by_danger.csv")
print(f"\nTop 10 by danger_removed:")
print(df_by_danger[["n_cells", "path_km", "avg_path_danger",
                     "danger_removed", "coverage_km2"]].head(10))

Saved 200 paths to drone_paths_by_danger.csv

Top 10 by danger_removed:
   n_cells  path_km  avg_path_danger  danger_removed  coverage_km2
0        6     30.0         0.740327        2.243865         150.0
1        6     30.0         0.735388        2.243865         150.0
2        6     30.0         0.753514        2.243865         150.0
3        6     30.0         0.761860        2.243865         150.0
4        6     30.0         0.750084        2.243865         150.0
5        6     30.0         0.745970        2.243865         150.0
6        6     30.0         0.584695        2.243865         150.0
7        6     30.0         0.650056        2.243865         150.0
8        6     30.0         0.755973        2.243865         150.0
9        6     30.0         0.751707        2.243865         150.0


    parser = argparse.ArgumentParser()
    #parser.add_argument("--filtered", default="drone_paths_by_danger.csv")
    #parser.add_argument("--filtered", default="human_paths_filtered.csv")
    parser.add_argument("--filtered", default="drone_paths_filtered.csv")
    parser.add_argument("--fire",     default="fire_risk_5km.csv")
    parser.add_argument("--animal",   default="animal_value_5km.csv")
    parser.add_argument("--out",      default="drone_PATHS_plot.png")
    #parser.add_argument("--out",      default="human_PATHS_plot.png")
    args = parser.parse_args()